# Training Model Klasifikasi Dokumen ALMEX
Train model `jenis_pipeline` dan `arah_pipeline` dari dataset PDF.

## Struktur Dataset
```
dataset/                     ← folder ini di repo root
├── PurchaseOrder/           (*.pdf)
├── Invoice/                 (*.pdf)
├── Penawaran/               (*.pdf)
├── SalesOrder/              (*.pdf)
└── SuratJalan/              (*.pdf)
```
**Label arah** di-derive otomatis dari nama folder jenis.
| Jenis | Arah |
|---|---|
| PurchaseOrder | Masuk |
| Invoice, Penawaran, SalesOrder, SuratJalan | Keluar |


In [ ]:
import os, re, warnings, random, glob, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import fitz  # PyMuPDF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# Path
DATASET_DIR = Path('./dataset')
MODEL_DIR = Path('../backend/ml_model')

# Arah mapping (HARUS sama dengan backend)
JENIS_KE_ARAH = {
    'PurchaseOrder': 'Masuk',
    'Invoice': 'Keluar',
    'Penawaran': 'Keluar',
    'SalesOrder': 'Keluar',
    'SuratJalan': 'Keluar',
}


## 1. Extract Text dari PDF

In [ ]:
def extract_text(file_path):
    """Extract text dari PDF pakai PyMuPDF. Fallback OCR kalau halaman kosong."""
    ext = os.path.splitext(file_path)[1].lower()
    text = ''
    if ext == '.pdf':
        doc = fitz.open(file_path)
        for page in doc:
            page_text = page.get_text().strip()
            if page_text:
                text += page_text + '\n'
            else:
                # Fallback OCR (butuh Tesseract di PATH)
                try:
                    import pytesseract
                    from PIL import Image
                    import cv2
                    pix = page.get_pixmap(dpi=200)
                    img = cv2.imdecode(np.frombuffer(pix.tobytes('png'), np.uint8), cv2.IMREAD_COLOR)
                    if img is not None:
                        img_rgb = Image.fromarray(img[:, :, ::-1])
                        ocr = pytesseract.image_to_string(img_rgb, lang='ind').strip()
                        if ocr:
                            text += ocr + '\n'
                except Exception:
                    pass
        doc.close()
    return text.strip()

print('Contoh extract satu file (ganti path):')
# sample = list(DATASET_DIR.glob('*/*.pdf'))[0]
# print(sample, '->', extract_text(str(sample))[:200])

## 2. Load Dataset

In [ ]:
data = []
for folder in sorted(os.listdir(DATASET_DIR)):
    folder_path = DATASET_DIR / folder
    if not folder_path.is_dir():
        continue
    jenis = folder.replace(' ', '')
    arah = JENIS_KE_ARAH.get(jenis, 'Keluar')
    pdf_files = [f for f in os.listdir(folder_path) if f.lower().endswith('.pdf')]
    for f in sorted(pdf_files):
        text = extract_text(str(folder_path / f))
        if text:
            data.append({'file': f, 'text': text, 'jenis': jenis, 'arah': arah})
            print(f'  [OK] {f}')
        else:
            print(f'  [EMPTY] {f} -> SKIP')

df = pd.DataFrame(data)
print(f'\nTotal terbaca: {len(df)} dokumen')
print(df['jenis'].value_counts())

## 3. Preprocessing
**HARUS IDENTIK** dengan `backend/ml/classifier.py`

In [ ]:
stemmer = StemmerFactory().create_stemmer()
stopwords = set(StopWordRemoverFactory().get_stop_words())
company_sw = {'pt', 'cv', 'tbk', 'abt', 'vi', 'nomor', 'perihal', 'lampiran', 'kepada', 'yth'}
domain_sw = {
    'rucika', 'pcs', 'batang', 'total', 'harga', 'diskon', 'tanggal', 'kode',
    'barang', 'nama', 'qty', 'satuan', 'rupiah', 'ribu', 'juta', 'indonesia',
    'tangerang', 'banten', 'kota', 'green', 'lake', 'city', 'ruko', 'timur',
    'bintang', 'almex', 'jan', 'feb', 'mar', 'apr', 'mei', 'jun', 'jul', 'agu',
    'sep', 'okt', 'nov', 'des', 'no', 'jumlah', 'sub', 'lain', 'biaya', 'ppn',
    'dpp', 'net', 'cash', 'transfer', 'dibuat', 'disetujui', 'pengirim',
    'penerima', 'keterangan',
}
all_stopwords = stopwords | company_sw | domain_sw

def preprocess(text):
    text = text.lower()
    text = re.sub(r'\b\d{4}\b', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [t for t in text.split() if t not in all_stopwords and len(t) > 2]
    if tokens:
        tokens = stemmer.stem(' '.join(tokens)).split()
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(preprocess)
df = df[df['clean_text'] != ''].reset_index(drop=True)
print(f'After cleaning: {len(df)} dokumen')
print(df['clean_text'].iloc[0][:300])

## 4. Train/Test Split

In [ ]:
X = df['clean_text']
y_arah = df['arah']
y_jenis = df['jenis']

X_train, X_test, y_train_arah, y_test_arah, y_train_jenis, y_test_jenis = train_test_split(
    X, y_arah, y_jenis, test_size=0.2, random_state=42, stratify=y_jenis
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

## 5. Build & Train Pipeline

In [ ]:
n_docs = len(df)
max_features = min(800, max(300, n_docs * 8))
min_df = 3 if n_docs >= 80 else 2
max_df = 0.70

pipe_arah = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=max_features, ngram_range=(1, 2),
                               min_df=min_df, max_df=max_df, sublinear_tf=True)),
    ('clf', MultinomialNB(alpha=1.0))
])
pipe_jenis = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=max_features, ngram_range=(1, 2),
                               min_df=min_df, max_df=max_df, sublinear_tf=True)),
    ('clf', MultinomialNB(alpha=1.0))
])

pipe_arah.fit(X_train, y_train_arah)
pipe_jenis.fit(X_train, y_train_jenis)

print('Training selesai.')

## 6. Evaluasi (Accuracy, F1, Classification Report)

In [ ]:
def evaluate(pipeline, X_test, y_test, title):
    y_pred = pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    print(f'\n=== {title} ===')
    print(f'Accuracy  : {acc:.4f}')
    print(f'Precision : {prec:.4f}')
    print(f'Recall    : {rec:.4f}')
    print(f'F1 Score  : {f1:.4f}')
    print(classification_report(y_test, y_pred, zero_division=0))
    return y_pred

pred_arah = evaluate(pipe_arah, X_test, y_test_arah, 'Arah Dokumen')
pred_jenis = evaluate(pipe_jenis, X_test, y_test_jenis, 'Jenis Dokumen')

## 7. Confusion Matrix

In [ ]:
def plot_cm(y_true, y_pred, labels, title):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.title(f'CM - {title}')
    plt.ylabel('Actual'); plt.xlabel('Predicted')
    plt.tight_layout()
    plt.show()

plot_cm(y_test_arah, pred_arah, sorted(set(y_test_arah)), 'Arah')
plot_cm(y_test_jenis, pred_jenis, sorted(set(y_test_jenis)), 'Jenis')

## 8. Cross Validation (5-Fold)

In [ ]:
cv_arah = cross_val_score(pipe_arah, X, y_arah,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), scoring='f1_weighted')
cv_jenis = cross_val_score(pipe_jenis, X, y_jenis,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), scoring='f1_weighted')

print(f'Arah  CV F1: {cv_arah.mean():.4f} (+/- {cv_arah.std():.4f})')
print(f'Jenis CV F1: {cv_jenis.mean():.4f} (+/- {cv_jenis.std():.4f})')

## 9. Robustness Testing
Simulasi: keyword dihapus, header dihilangkan, OCR error.

In [ ]:
KEYWORDS = {
    'PurchaseOrder': ['purchase order', 'po', 'pesanan', 'pengadaan', 'pemesanan', 'order', 'pesan'],
    'Invoice': ['invoice', 'faktur', 'tagihan', 'inv', 'pembayaran', 'bill', 'tagih'],
    'Penawaran': ['penawaran', 'harga', 'kerjasama', 'proposal', 'offer', 'quotation', 'tawar'],
    'SalesOrder': ['sales order', 'so', 'penjualan', 'delivery order', 'jual'],
    'SuratJalan': ['surat jalan', 'pengiriman', 'delivery', 'sj', 'pengantar', 'kirim'],
}

def remove_keywords(text, jenis):
    words = text.lower().split()
    kws = KEYWORDS.get(jenis, [])
    return ' '.join([w for w in words if not any(kw in w or w in kw for kw in kws)])

def remove_header(text, ratio=0.25):
    words = text.split()
    start = int(len(words) * ratio)
    return ' '.join(words[start:])

def ocr_error(text, rate=0.10):
    words = text.split()
    out = []
    for w in words:
        if random.random() < rate:
            c = random.choice(['del', 'swap', 'jumble'])
            if c == 'del': continue
            elif c == 'swap' and len(w) > 2:
                ch = list(w); i = random.randint(0, len(ch)-2)
                ch[i], ch[i+1] = ch[i+1], ch[i]
                out.append(''.join(ch))
            elif c == 'jumble' and len(w) > 3:
                ch = list(w); random.shuffle(ch); out.append(''.join(ch))
            else:
                out.append(w)
        else:
            out.append(w)
    return ' '.join(out)

# Preprocess skenario
df['clean_no_kw'] = df.apply(lambda r: preprocess(remove_keywords(r['text'], r['jenis'])), axis=1)
df['clean_no_header'] = df['text'].apply(lambda t: preprocess(remove_header(t, 0.25)))
df['clean_ocr'] = df['text'].apply(lambda t: preprocess(ocr_error(t)))

print(f'\n{"Skenario":<40} {"Acc Arah":<10} {"F1 Arah":<10} {"Acc Jenis":<10} {"F1 Jenis":<10}')
print('-' * 70)
for name, xa, xj in [
    ('A. Data Lengkap', df['clean_text'], df['clean_text']),
    ('B. Tanpa Kata Kunci', df['clean_no_kw'], df['clean_no_kw']),
    ('C. Hapus Header 25%', df['clean_no_header'], df['clean_no_header']),
    ('D. OCR Error 10%', df['clean_ocr'], df['clean_ocr']),
]:
    pa = pipe_arah.predict(xa)
    pj = pipe_jenis.predict(xj)
    print(f'{name:<40} {accuracy_score(df["arah"], pa):<10.4f} {f1_score(df["arah"], pa, average="weighted"):<10.4f} '
          f'{accuracy_score(df["jenis"], pj):<10.4f} {f1_score(df["jenis"], pj, average="weighted"):<10.4f}')

## 10. Save Model

In [ ]:
os.makedirs(MODEL_DIR, exist_ok=True)

# Refit pada full dataset untuk production model
pipe_arah.fit(X, y_arah)
pipe_jenis.fit(X, y_jenis)

joblib.dump(pipe_arah, MODEL_DIR / 'arah_pipeline.pkl')
joblib.dump(pipe_jenis, MODEL_DIR / 'jenis_pipeline.pkl')

# Hapus file lama kalau ada
old = MODEL_DIR / 'tfidf_vectorizer.pkl'
if old.exists(): old.unlink()

print(f'Saved to {MODEL_DIR}: {os.listdir(MODEL_DIR)}')